# Phase 4 — Inference

Run sliding window inference on validation/test cases and save predictions as NIfTI.

**Kernel:** `gtx-1080-IP`

In [ ]:
import sys, os
from pathlib import Path

PROJECT_ROOT = Path(os.getcwd()).parent if 'notebooks' in os.getcwd() else Path(os.getcwd())
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
print(f'Project root: {PROJECT_ROOT}')

## Configuration

In [ ]:
import torch
from src.utils.config import load_config
from src.utils.seed import set_seed

CONFIG = 'dev'
CHECKPOINT = 'checkpoints/best_model.pth'
SPLIT = 'val'  # 'val' or 'test'
NUM_CASES = None  # None = all cases in the split

cfg = load_config(CONFIG)
set_seed(cfg['seed'])

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Config: {CONFIG}')
print(f'Device: {device}')
print(f'Checkpoint: {CHECKPOINT}')

## Load Data Splits

In [ ]:
from src.data.dataset import discover_brats_samples
from src.data.splits import create_splits

samples = discover_brats_samples(cfg['paths']['data_root'])
_, val_samples, test_samples = create_splits(
    samples,
    ratios=cfg['data']['split_ratios'],
    seed=cfg['seed'],
    splits_dir=cfg['paths']['splits_dir'],
)

target_samples = val_samples if SPLIT == 'val' else test_samples
print(f'{SPLIT} samples: {len(target_samples)}')
print(f'Patient IDs: {[s["patient_id"] for s in target_samples]}')

## Run Sliding Window Inference

In [ ]:
from src.inference.predict import run_inference
from src.utils.config import PROJECT_ROOT

checkpoint_path = str(PROJECT_ROOT / CHECKPOINT)

results = run_inference(
    cfg=cfg,
    checkpoint_path=checkpoint_path,
    samples=target_samples,
    device=device,
    num_cases=NUM_CASES,
)

print(f'\n✅ {len(results)} predictions saved to {cfg["paths"]["predictions_dir"]}')

## Quick Preview

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

if results:
    r = results[0]
    pred = r['prediction']  # (3, D, H, W)
    gt = r['label']  # (3, D, H, W)
    
    # Find slice with most tumor
    tumor_sum = gt[0].sum(axis=(1, 2))
    center = int(np.argmax(tumor_sum)) if tumor_sum.max() > 0 else gt.shape[1] // 2
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    
    # GT
    gt_rgb = np.zeros((*gt.shape[2:], 3))
    gt_rgb[..., 1] = gt[0, center]  # WT green
    gt_rgb[..., 2] = gt[1, center]  # TC blue
    gt_rgb[..., 0] = gt[2, center]  # ET red
    axes[0].imshow(gt_rgb)
    axes[0].set_title(f'Ground Truth — {r["patient_id"]}')
    axes[0].axis('off')
    
    # Pred
    pred_rgb = np.zeros((*pred.shape[2:], 3))
    pred_rgb[..., 1] = pred[0, center]
    pred_rgb[..., 2] = pred[1, center]
    pred_rgb[..., 0] = pred[2, center]
    axes[1].imshow(pred_rgb)
    axes[1].set_title('Prediction')
    axes[1].axis('off')
    
    plt.tight_layout()
    plt.show()